# 02. Exploratory Data Analysis & Statistical Hypothesis Testing
## Banking Customer Churn Analytics

### Objectives
1. Perform bivariate and multivariate exploratory analysis across demographic and financial dimensions.
2. Conduct formal statistical hypothesis tests:
   - **Chi-Square Tests of Independence** & **Cramer's V** for categorical features (`Geography`, `Gender`, `NumOfProducts`, `HasCrCard`, `IsActiveMember`).
   - **Mann-Whitney U** and **Independent Two-Sample T-Tests** with **Cohen's d** effect sizes for numerical features (`CreditScore`, `Age`, `Tenure`, `Balance`, `EstimatedSalary`).
3. Identify core behavioral and structural drivers behind customer attrition.


In [1]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath(".."))
from src.eda import BankChurnEDA
from src.visualization import BankVisualizer

sns.set_theme(style="whitegrid")
df = pd.read_csv("../data/processed/bank_customer_churn_clean.csv")
print(f"Loaded Clean Dataset: {df.shape[0]:,} rows, {df.shape[1]} columns")


Loaded Clean Dataset: 10,000 rows, 14 columns


## 1. Overall Churn Distribution

In [2]:
churn_counts = df['Exited'].value_counts()
churn_pct = df['Exited'].value_counts(normalize=True) * 100

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(x=churn_counts.index, y=churn_counts.values, hue=churn_counts.index, palette=["#2B6CB0", "#E53E3E"], legend=False, ax=ax[0])
ax[0].set_title("Customer Volume by Churn Status", fontweight="bold")
ax[0].set_xticks([0, 1])
ax[0].set_xticklabels(["Retained (0)", "Churned (1)"])
ax[0].set_ylabel("Customer Count")

ax[1].pie(churn_pct, labels=["Retained", "Churned"], autopct="%1.1f%%", colors=["#2B6CB0", "#E53E3E"], explode=(0, 0.08), startangle=140)
ax[1].set_title("Overall Churn Proportion", fontweight="bold")
plt.tight_layout()
plt.show()


findfont: Failed to find font weight semibold, now using 700.


C:\Users\asus\AppData\Local\Temp\ipykernel_24424\1697404765.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Categorical Drivers & Statistical Hypothesis Testing (Chi-Square & Cramer's V)

In [3]:
eda = BankChurnEDA(df)
cat_results = eda.analyze_categorical_features()

summary_records = []
for col, res in cat_results.items():
    summary_records.append({
        "Feature": col,
        "Cramer_V": res["cramers_v"],
        "Chi2_Stat": res["chi2_statistic"],
        "p_value": res["p_value"],
        "Statistically_Significant": res["is_statistically_significant"]
    })

pd.DataFrame(summary_records).sort_values(by="Cramer_V", ascending=False)


,Feature,Cramer_V,Chi2_Stat,p_value,Statistically_Significant
2,NumOfProducts,0.3878,1503.6294,0.000000e+00,True
0,Geography,0.1736,301.2553,3.830318e-66,True
4,IsActiveMember,0.1559,242.9853,8.785858e-55,True
1,Gender,0.1063,112.9186,2.248210e-26,True
3,HasCrCard,0.0069,0.4713,4.923724e-01,False


## 3. Visualizing Key Categorical Drivers

In [4]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))

# Geography
geo_df = df.groupby("Geography", as_index=False)["Exited"].mean()
sns.barplot(data=geo_df, x="Geography", y="Exited", hue="Geography", palette="Blues_r", legend=False, ax=axes[0, 0])
axes[0, 0].set_title("Churn Rate by Geography (Germany has ~2x churn)", fontweight="bold")
axes[0, 0].set_ylabel("Churn Rate")

# Products
prod_df = df.groupby("NumOfProducts", as_index=False)["Exited"].mean()
sns.barplot(data=prod_df, x="NumOfProducts", y="Exited", hue="NumOfProducts", palette="Reds", legend=False, ax=axes[0, 1])
axes[0, 1].set_title("Churn Rate by Number of Products (Severe defection with 3+ products)", fontweight="bold")
axes[0, 1].set_ylabel("Churn Rate")

# IsActiveMember
act_df = df.groupby("IsActiveMember", as_index=False)["Exited"].mean()
sns.barplot(data=act_df, x="IsActiveMember", y="Exited", hue="IsActiveMember", palette="Set2", legend=False, ax=axes[1, 0])
axes[1, 0].set_title("Churn Rate by Active Membership Status", fontweight="bold")
axes[1, 0].set_xticks([0, 1])
axes[1, 0].set_xticklabels(["Inactive (0)", "Active (1)"])
axes[1, 0].set_ylabel("Churn Rate")

# Gender
gen_df = df.groupby("Gender", as_index=False)["Exited"].mean()
sns.barplot(data=gen_df, x="Gender", y="Exited", hue="Gender", palette="coolwarm", legend=False, ax=axes[1, 1])
axes[1, 1].set_title("Churn Rate by Customer Gender", fontweight="bold")
axes[1, 1].set_ylabel("Churn Rate")

plt.tight_layout()
plt.show()


C:\Users\asus\AppData\Local\Temp\ipykernel_24424\1373964570.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Numerical Drivers & Hypothesis Testing (Mann-Whitney U & Cohen's d)

In [5]:
num_results = eda.analyze_numerical_features()
num_summary = []
for col, res in num_results.items():
    num_summary.append({
        "Feature": col,
        "Cohen_d": res["cohens_d"],
        "Churned_Mean": res["churned_mean"],
        "Retained_Mean": res["retained_mean"],
        "MWU_p_val": res["mwu_p_value"],
        "Statistically_Significant": res["is_statistically_significant"]
    })

pd.DataFrame(num_summary).sort_values(by="Cohen_d", key=abs, ascending=False)


,Feature,Cohen_d,Churned_Mean,Retained_Mean,MWU_p_val,Statistically_Significant
1,Age,0.7391,44.84,37.41,3.210401e-230,True
3,Balance,0.2964,91108.54,72745.30,1.115840e-28,True
0,CreditScore,-0.0673,645.35,651.85,1.986866e-02,True
2,Tenure,-0.0348,4.93,5.03,1.621938e-01,False
4,EstimatedSalary,0.0300,101465.68,99738.39,2.270515e-01,False


## 5. Numerical Feature Distributions (KDE Density Plots)

In [6]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
palette = {0: "#2B6CB0", 1: "#E53E3E"}

# Age
sns.kdeplot(data=df, x="Age", hue="Exited", common_norm=False, fill=True, palette=palette, alpha=0.35, ax=axes[0])
axes[0].set_title("Age Distribution (Peak defection between 45-60)", fontweight="bold")

# Balance
sns.kdeplot(data=df[df["Balance"] > 0], x="Balance", hue="Exited", common_norm=False, fill=True, palette=palette, alpha=0.35, ax=axes[1])
axes[1].set_title("Account Balance Distribution (> $0)", fontweight="bold")

# Credit Score
sns.kdeplot(data=df, x="CreditScore", hue="Exited", common_norm=False, fill=True, palette=palette, alpha=0.35, ax=axes[2])
axes[2].set_title("Credit Score Distribution", fontweight="bold")

plt.tight_layout()
plt.show()


C:\Users\asus\AppData\Local\Temp\ipykernel_24424\4144247327.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Correlation Heatmap & Feature Interdependence

In [7]:
corr_mat = eda.get_correlation_matrix()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_mat, dtype=bool))
sns.heatmap(corr_mat, mask=mask, annot=True, fmt=".2f", cmap="vlag", center=0, square=True, linewidths=0.5)
plt.title("Correlation Matrix of Banking Features", fontweight="bold", pad=12)
plt.tight_layout()
plt.show()


C:\Users\asus\AppData\Local\Temp\ipykernel_24424\332413566.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
